# 🧠 BhashaLens — Notebook 2: MarianMT Training

Fine-tunes MarianMT models for each language direction using cleaned datasets from Notebook 1.

**Architecture:** 6 encoder/decoder layers, 512 hidden, 8 attention heads, 32K vocab

**Training:** Batch 32, LR 0.0003, warmup 4000 steps, max 50K steps, early stopping

**Checkpoint Resume:** Saves to local disk for resume across sessions

---

## 1. Setup & Dependencies

In [1]:
%pip install -q transformers[torch] sentencepiece sacrebleu datasets accelerate pyyaml python-dotenv sacremoses

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

import os
import json
import time
import shutil
from pathlib import Path
from datetime import datetime

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from transformers import (
    MarianMTModel,
    MarianTokenizer,
    MarianConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from datasets import Dataset
import sacrebleu

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

d:\BHASHALENS1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.10.0+cpu
CUDA available: False


In [3]:
# ── Local Paths ──
BASE_DIR = Path('./bhashalens_ml')

DATA_CLEANED = BASE_DIR / 'data' / 'cleaned'
CHECKPOINTS_DIR = BASE_DIR / 'checkpoints'
MODELS_DIR = BASE_DIR / 'models' / 'trained'
REPORTS_DIR = BASE_DIR / 'reports'

for d in [CHECKPOINTS_DIR, MODELS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Data from: {DATA_CLEANED}')
print(f'Checkpoints: {CHECKPOINTS_DIR}')
print(f'Models: {MODELS_DIR}')


Data from: bhashalens_ml\data\cleaned
Checkpoints: bhashalens_ml\checkpoints
Models: bhashalens_ml\models\trained


## 2. Configuration

In [4]:
# ── Training Configuration (from spec) ──
TRAINING_CONFIG = {
    # Which language pairs to train (set to one pair for a single session)
    'language_pairs': [
        'hi-en',   # Hindi → English
        'en-hi',   # English → Hindi
        'mr-en',   # Marathi → English
        'en-mr',   # English → Marathi
        'ta-en',   # Tamil → English
        'en-ta',   # English → Tamil
        'gu-en',   # Gujarati → English
        'en-gu',   # English → Gujarati
    ],
    
    # HuggingFace base models to fine-tune from
    'base_models': {
        'hi-en': 'Helsinki-NLP/opus-mt-hi-en',
        'en-hi': 'Helsinki-NLP/opus-mt-en-hi',
        'mr-en': 'Helsinki-NLP/opus-mt-mr-en',
        'en-mr': 'Helsinki-NLP/opus-mt-en-mr',
        'ta-en': 'Helsinki-NLP/opus-mt-ta-en',
        'en-ta': 'Helsinki-NLP/opus-mt-en-ta',
        'gu-en': 'Helsinki-NLP/opus-mt-gu-en',
        'en-gu': 'Helsinki-NLP/opus-mt-en-gu',
    },
    
    # Hyperparameters
    'batch_size': 32,
    'learning_rate': 3e-4,
    'warmup_steps': 4000,
    'max_steps': 50000,
    'eval_steps': 2500,
    'save_steps': 5000,
    'early_stopping_patience': 4,  # 4 evals * 2500 = 10000 steps
    'label_smoothing': 0.1,
    'max_length': 128,
    'gradient_accumulation_steps': 2,  # effective batch = 64
    'fp16': torch.cuda.is_available(),
    
    # Quality thresholds
    'bleu_thresholds': {
        'hi-en': 25.0, 'en-hi': 25.0,
        'mr-en': 20.0, 'en-mr': 20.0,
        'ta-en': 20.0, 'en-ta': 20.0,
        'gu-en': 20.0, 'en-gu': 20.0,
    },
}

# ⚠️ PICK WHICH PAIR(S) TO TRAIN THIS SESSION
# If you have a GPU, you can train all pairs. Otherwise, train 1-2 at a time.
PAIRS_THIS_SESSION = ['hi-en']  # Change this each session

print(f'Training {len(PAIRS_THIS_SESSION)} pair(s) this session: {PAIRS_THIS_SESSION}')

Training 1 pair(s) this session: ['hi-en']


## 3. Load Cleaned Data

In [5]:
def load_data(lang_pair: str) -> dict:
    """Load cleaned train/val/test splits from Notebook 1 outputs."""
    data_path = DATA_CLEANED / lang_pair
    
    if not data_path.exists():
        raise FileNotFoundError(
            f'No cleaned data for {lang_pair} at {data_path}.\n'
            f'Run Notebook 1 first, or upload data to this path.'
        )
    
    splits = {}
    for split_name in ['train', 'val', 'test']:
        fpath = data_path / f'{split_name}.tsv'
        df = pd.read_csv(fpath, sep='\t')
        splits[split_name] = df
        print(f'  {lang_pair} {split_name}: {len(df):,} pairs')
    
    return splits

# Load data for selected pairs
all_data = {}
for lp in PAIRS_THIS_SESSION:
    print(f'\nLoading {lp}...')
    all_data[lp] = load_data(lp)


Loading hi-en...
  hi-en train: 1,812,819 pairs
  hi-en val: 100,939 pairs
  hi-en test: 100,854 pairs


## 4. Tokenization & Dataset Preparation

In [6]:
def prepare_dataset(df: pd.DataFrame, tokenizer, max_length: int = 128) -> Dataset:
    """Convert DataFrame to HuggingFace Dataset with tokenization."""
    
    def tokenize_fn(examples):
        model_inputs = tokenizer(
            examples['source'],
            max_length=max_length,
            truncation=True,
            padding='max_length',
        )
        
        labels = tokenizer(
            text_target=examples['target'],
            max_length=max_length,
            truncation=True,
            padding='max_length',
        )
        
        model_inputs['labels'] = labels['input_ids']
        return model_inputs
    
    dataset = Dataset.from_pandas(df[['source', 'target']])
    tokenized = dataset.map(
        tokenize_fn,
        batched=True,
        batch_size=1000,
        remove_columns=['source', 'target'],
        desc='Tokenizing',
    )
    
    return tokenized

print('Dataset preparation function ready.')

Dataset preparation function ready.


## 5. BLEU Evaluation Metric

In [7]:
def compute_bleu_metrics(eval_preds, tokenizer):
    """Compute BLEU score for evaluation."""
    preds, labels = eval_preds
    
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # Replace -100 with pad token id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Clean up
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    
    result = sacrebleu.corpus_bleu(
        decoded_preds,
        [decoded_labels],
    )
    
    return {'bleu': round(result.score, 2)}

print('BLEU metric function ready.')

BLEU metric function ready.


## 6. Training Loop

In [8]:
def train_model(lang_pair: str, data: dict, config: dict) -> dict:
    """Train a MarianMT model for a single language direction."""
    print(f'\n{"="*60}')
    print(f'Training: {lang_pair}')
    print(f'{"="*60}')
    
    base_model_name = config['base_models'].get(lang_pair)
    if not base_model_name:
        print(f'No base model configured for {lang_pair}, skipping.')
        return None
    
    checkpoint_dir = CHECKPOINTS_DIR / lang_pair
    output_dir = MODELS_DIR / lang_pair
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Check for existing checkpoint to resume from
    resume_from = None
    if checkpoint_dir.exists():
        existing_checkpoints = sorted(
            [d for d in checkpoint_dir.iterdir() if d.is_dir() and 'checkpoint' in d.name],
            key=lambda x: int(x.name.split('-')[-1]) if x.name.split('-')[-1].isdigit() else 0
        )
        if existing_checkpoints:
            resume_from = str(existing_checkpoints[-1])
            print(f'📂 Resuming from checkpoint: {resume_from}')
    
    # Load model and tokenizer
    print(f'Loading base model: {base_model_name}')
    tokenizer = MarianTokenizer.from_pretrained(base_model_name)
    model = MarianMTModel.from_pretrained(base_model_name)
    
    print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
    
    # Prepare datasets
    print('Preparing datasets...')
    train_dataset = prepare_dataset(data['train'], tokenizer, config['max_length'])
    val_dataset = prepare_dataset(data['val'], tokenizer, config['max_length'])
    
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
    )
    
    # Training arguments
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(checkpoint_dir),
        
        # Core hyperparameters
        per_device_train_batch_size=config['batch_size'],
        per_device_eval_batch_size=config['batch_size'],
        gradient_accumulation_steps=config['gradient_accumulation_steps'],
        learning_rate=config['learning_rate'],
        warmup_steps=config['warmup_steps'],
        max_steps=config['max_steps'],
        label_smoothing_factor=config['label_smoothing'],
        fp16=config['fp16'],
        
        # Evaluation & checkpointing
        eval_strategy='steps',
        eval_steps=config['eval_steps'],
        save_strategy='steps',
        save_steps=config['save_steps'],
        save_total_limit=3,
        load_best_model_at_end=True,
        metric_for_best_model='bleu',
        greater_is_better=True,
        
        # Generation settings for eval
        predict_with_generate=True,
        generation_max_length=config['max_length'],
        
        # Logging
        logging_dir=str(checkpoint_dir / 'logs'),
        logging_steps=100,
        report_to='none',
        
        # Misc
        seed=42,
        dataloader_num_workers=2,
        remove_unused_columns=False,
    )
    
    # Create compute_metrics with closure over tokenizer
    def compute_metrics(eval_preds):
        return compute_bleu_metrics(eval_preds, tokenizer)
    
    # Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config['early_stopping_patience'])],
    )
    
    # Train
    start_time = time.time()
    print(f'\n🚀 Starting training... (max {config["max_steps"]:,} steps)')
    
    train_result = trainer.train(resume_from_checkpoint=resume_from)
    
    training_time = (time.time() - start_time) / 3600
    print(f'\n⏱️ Training completed in {training_time:.1f} hours')
    
    # Save final model
    print(f'💾 Saving model to {output_dir}...')
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))
    
    # Save training history
    history = {
        'lang_pair': lang_pair,
        'base_model': base_model_name,
        'training_time_hours': round(training_time, 2),
        'train_loss': train_result.training_loss,
        'train_steps': train_result.global_step,
        'log_history': trainer.state.log_history,
        'timestamp': datetime.now().isoformat(),
    }
    
    with open(output_dir / 'training_history.json', 'w') as f:
        json.dump(history, f, indent=2, default=str)
    
    return {
        'trainer': trainer,
        'tokenizer': tokenizer,
        'model': model,
        'history': history,
    }

print('Training function ready.')

Training function ready.


In [9]:
# ── Run Training ──
trained_models = {}

for lp in PAIRS_THIS_SESSION:
    result = train_model(lp, all_data[lp], TRAINING_CONFIG)
    if result:
        trained_models[lp] = result


Training: hi-en
Loading base model: Helsinki-NLP/opus-mt-hi-en


d:\BHASHALENS1\.venv\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Loading weights: 100%|██████████| 258/258 [00:00<00:00, 865.47it/s, Materializing param=model.shared.weight]                                  
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model parameters: 138,553,856
Preparing datasets...


Tokenizing:   7%|▋         | 123000/1812819 [00:43<10:01, 2809.28 examples/s]


ValueError: Input must be a string, list of strings, or list of ints, got: <class 'NoneType'>

## 7. Final Evaluation

In [ ]:
def evaluate_model(lang_pair: str, result: dict, test_data: pd.DataFrame) -> dict:
    """Final evaluation on test set with quality gate."""
    print(f'\n📊 Evaluating {lang_pair} on test set...')
    
    trainer = result['trainer']
    tokenizer = result['tokenizer']
    model = result['model']
    
    # Prepare test dataset
    test_dataset = prepare_dataset(test_data, tokenizer, TRAINING_CONFIG['max_length'])
    
    # Evaluate
    eval_results = trainer.evaluate(eval_dataset=test_dataset)
    test_bleu = eval_results.get('eval_bleu', 0)
    
    print(f'  Test BLEU: {test_bleu:.2f}')
    print(f'  Test Loss: {eval_results.get("eval_loss", "N/A")}')
    
    # Quality gate
    threshold = TRAINING_CONFIG['bleu_thresholds'].get(lang_pair, 20.0)
    passed = test_bleu >= threshold
    status = '✅ PASSED' if passed else '❌ FAILED'
    print(f'  Quality Gate ({threshold:.0f} BLEU): {status}')
    
    # Generate sample translations
    print(f'\n  Sample Translations:')
    sample_df = test_data.sample(n=min(10, len(test_data)), random_state=42)
    samples = []
    
    model.eval()
    device = next(model.parameters()).device
    
    for _, row in sample_df.iterrows():
        inputs = tokenizer(row['source'], return_tensors='pt', padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=128, num_beams=4)
        
        translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        sample = {
            'source': row['source'],
            'reference': row['target'],
            'hypothesis': translation,
        }
        samples.append(sample)
        print(f'    SRC: {row["source"][:80]}')
        print(f'    REF: {row["target"][:80]}')
        print(f'    HYP: {translation[:80]}')
        print()
    
    # Measure inference speed
    print('  Measuring inference speed...')
    speed_samples = test_data.sample(n=min(100, len(test_data)), random_state=42)
    start = time.time()
    
    for _, row in speed_samples.iterrows():
        inputs = tokenizer(row['source'], return_tensors='pt', padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            model.generate(**inputs, max_length=128, num_beams=4)
    
    elapsed = time.time() - start
    sps = len(speed_samples) / elapsed
    ms_per_sentence = (elapsed / len(speed_samples)) * 1000
    print(f'  Speed: {sps:.1f} sentences/sec ({ms_per_sentence:.0f} ms/sentence on GPU)')
    
    # Save evaluation report
    eval_report = {
        'lang_pair': lang_pair,
        'test_bleu': test_bleu,
        'test_loss': eval_results.get('eval_loss'),
        'quality_threshold': threshold,
        'quality_passed': passed,
        'sentences_per_second': round(sps, 1),
        'ms_per_sentence': round(ms_per_sentence, 0),
        'sample_translations': samples,
        'timestamp': datetime.now().isoformat(),
    }
    
    report_path = REPORTS_DIR / f'eval_{lang_pair}.json'
    with open(report_path, 'w') as f:
        json.dump(eval_report, f, indent=2, ensure_ascii=False)
    
    print(f'  Report saved to: {report_path}')
    return eval_report

# Run evaluation for all trained models
eval_reports = {}
for lp in PAIRS_THIS_SESSION:
    if lp in trained_models:
        report = evaluate_model(lp, trained_models[lp], all_data[lp]['test'])
        eval_reports[lp] = report

## 8. Training Summary

In [ ]:
print('\n' + '='*60)
print('📋 TRAINING SUMMARY')
print('='*60)

for lp, report in eval_reports.items():
    status = '✅' if report['quality_passed'] else '❌'
    print(f'\n{status} {lp}:')
    print(f'   BLEU: {report["test_bleu"]:.2f} (threshold: {report["quality_threshold"]:.0f})')
    print(f'   Speed: {report["sentences_per_second"]:.1f} sent/sec')
    print(f'   Model saved at: {MODELS_DIR / lp}')

# List all trained model directories
print(f'\n\n📁 Trained Models Directory: {MODELS_DIR}')
if MODELS_DIR.exists():
    for d in sorted(MODELS_DIR.iterdir()):
        if d.is_dir():
            size_mb = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 1e6
            print(f'   {d.name}: {size_mb:.0f} MB')

print('\n\n🔜 Next: Run Notebook 3 (Quantize & ONNX Export) on these models.')

## 9. Session Resume Helper

If training was interrupted, change `PAIRS_THIS_SESSION` in Cell 2.1 to the next pair and re-run.

Training will automatically resume from the last checkpoint saved to disk.

In [ ]:
# Show checkpoint status for all pairs
print('📂 Checkpoint Status:')
all_pairs = TRAINING_CONFIG['language_pairs']

for lp in all_pairs:
    ckpt_dir = CHECKPOINTS_DIR / lp
    model_dir = MODELS_DIR / lp
    
    if model_dir.exists() and (model_dir / 'config.json').exists():
        print(f'  ✅ {lp}: Training COMPLETE')
    elif ckpt_dir.exists():
        checkpoints = sorted(
            [d for d in ckpt_dir.iterdir() if d.is_dir() and 'checkpoint' in d.name],
            key=lambda x: int(x.name.split('-')[-1]) if x.name.split('-')[-1].isdigit() else 0
        )
        if checkpoints:
            last_step = checkpoints[-1].name.split('-')[-1]
            print(f'  🔄 {lp}: In progress (last checkpoint: step {last_step})')
        else:
            print(f'  ⏳ {lp}: Not started')
    else:
        print(f'  ⏳ {lp}: Not started')